# Per-dataset sensitivity analyses (CCM revision)

Primary results in the paper are per-dataset; this notebook adds the per-dataset
sensitivity analyses requested in review, using **each dataset's own preprocessing**
(imported from its `*Util.py`).

| Analysis | Reviewer point | Datasets |
|---|---|---|
| IPW-weighted interaction test | R1 Major 1 | eICU, PMAP, MIMIC-IV (observational) |
| Lasso S-learner (penalized interactions) | R1 Major 2 | all 4 |
| GATES for neurologic outcome | R1 minor h | all 4 |
| Uniform |r|>0.7 TTM-collinearity filter | R1 minor d | eICU (confirm unchanged) |

Exposure definitions unchanged: eICU `treatment_hypothermia`; PMAP <36°C ≥12 consecutive h;
MIMIC-IV cooling device ≥12 consecutive h; HYPERION randomization.

Run on the cluster from this folder:
```bash
jupyter nbconvert --to notebook --execute perDatasetSensitivity.ipynb \
  --output perDatasetSensitivity.ipynb --ExecutePreprocessor.timeout=-1
```
Writes `per_dataset_results.csv` for the response-letter `[INSERT]` slots.


In [ ]:
# CONFIG
SEED = 42
EVAL_METHOD = 'crossfit'   # 'crossfit' = k-fold out-of-fold prediction (robust, full n);
                           # 'split'    = single stratified 70/30 train/test (matches the main paper)
N_SPLITS = 5               # folds for cross-fitting
TEST_SIZE = 0.30           # used when EVAL_METHOD == 'split'
N_GATES_GROUPS = 5
PS_CLIP = (0.05, 0.95)
CORR_THRESHOLD = 0.7
# Require the last mGCS to be measured >= this many time-units after the first, to avoid
# baseline≈outcome leakage in the neurologic analysis (last motor GCS ~ first motor GCS when
# measured in a short window). Units are each dataset's own (eICU: minutes from admission);
# 0 keeps the current behavior (just different timestamps). Tune per dataset and read the
# printed exclusion counts.
MIN_MGCS_GAP = {'eICU': 0, 'PMAP': 0, 'MIMIC-IV': 0}
REPO_ROOT = os.path.abspath('..')

# Predictor CSVs (the filenames the *AnalysisDML notebooks actually read).
CSV = {   # the CSVs the *AnalysisDML notebooks actually read
    'eICU':     os.path.join(REPO_ROOT, 'eICU',     'eICUPredictorsDiag.csv'),
    'PMAP':     os.path.join(REPO_ROOT, 'pmap',     'PMAP_Predictors2.csv'),
    'MIMIC-IV': os.path.join(REPO_ROOT, 'mimiciv',  'MIMIC_Predictors.csv'),
    'HYPERION': os.path.join(REPO_ROOT, 'hyperion', 'predictorsDf.csv'),
}
DATASETS = [
    dict(name='eICU',     observational=True),
    dict(name='PMAP',     observational=True),
    dict(name='MIMIC-IV', observational=True),
    dict(name='HYPERION', observational=False),  # randomized: no IPW, unweighted GATES (Fig 3)
]

# Curated feature lists = the 'columns' each *AnalysisDML notebook trains on. Restricting to
# these (~30-40) instead of the ~1600 raw predictor columns is what keeps CausalForest and the
# lasso from overfitting (Reviewer 1 Major 2) and makes the lasso count meaningful.
CURATED = {
 'eICU': ['gender','age','bmi',
    'nurse_first_Non-Invasive BP Systolic','nurse_first_Non-Invasive BP Diastolic',
    'nurse_first_Non-Invasive BP Mean','nurse_first_Heart Rate','nurse_first_O2 Saturation',
    'lab_first_Respiratory Rate','lab_first_FiO2','nurse_first_GCS Total','nurse_first_Motor',
    'nurse_first_QTc','lab_first_pH','lab_first_paO2','lab_first_paCO2','lab_first_bicarbonate',
    'lab_first_lactate','lab_first_WBC x 1000','lab_first_Hgb','lab_first_platelets x 1000',
    'lab_first_sodium','lab_first_potassium','lab_first_BUN','lab_first_creatinine',
    'lab_first_calcium','lab_first_magnesium','lab_first_glucose','lab_first_troponin - T',
    'diagnosis_initial rhythm: ventricular fibrillation',
    'diagnosis_initial rhythm: ventricular tachycardia',
    'diagnosis_initial rhythm: pulseless electrical activity',
    'diagnosis_initial rhythm: asystole'],
 'PMAP': ['gender','age','first_mGCS','flo_first_r_cpn_glasgow_coma_scale_score',
    'flo_first_bp_systolic','flo_first_bp_diastolic','flo_first_r_map',
    'flo_first_r_ed_pre-arrival_pulse_(heart_rate)','flo_first_r_sao2','flo_first_r_fio2',
    'flo_first_r_sofa_score','flo_first_r_bmi','flo_first_r_pao2','flo_first_r_paco2',
    'flo_first_r_resp_ph','lab_first_lactate','lab_first_troponin','lab_first_hemoglobin',
    'lab_first_platelet_count','lab_first_creatinine,whole_blood','lab_first_glucose,whole_blood',
    'lab_first_potassium,whole_blood','lab_first_sodium,whole_blood','lab_first_calcium,_serum',
    'lab_first_magnesium','asystole','pea','VF'],
 'MIMIC-IV': ['gender','age','bmi','first_mGCS',
    'chart_first_heart_rate','chart_first_o2_saturation_pulseoxymetry','chart_first_respiratory_rate',
    'chart_first_fio2_(ch)','chart_first_non_invasive_blood_pressure_systolic',
    'chart_first_non_invasive_blood_pressure_diastolic','chart_first_non_invasive_blood_pressure_mean',
    'chart_first_ph_(arterial)','chart_first_arterial_o2_pressure','chart_first_arterial_co2_pressure',
    'chart_first_lactic_acid','chart_first_wbc','chart_first_hemoglobin','lab_first_platelet_count',
    'chart_first_sodium_(serum)','lab_first_potassium_(serum)','chart_first_bun',
    'chart_first_creatinine_(serum)','chart_first_calcium_non-ionized','chart_first_magnesium',
    'chart_first_glucose_(serum)','lab_first_troponin-t','chart_first_qtc',
    'long_title_ventricular_fibrillation'],
 'HYPERION': ['J0_AGE','J0_SEX','J0_BMI','J0_PAS','J0_PAD','J0_PAM','J0_FC','J0_SPO2',
    'J0_GLASGOW','J0_MOTRICE','J0_RYTHM','J0_NOFLOW','J0_LOWFLOW','J0_IGSII',
    'BIO_LEUCO','BIO_HEMO','BIO_PLAQ','BIO_SODIUM','BIO_POTAS','BIO_UREE','BIO_CREAT',
    'BIO_CALCIUM','BIO_MAGNE','BIO_GLYCEMI','BIO_LACTAT','BIO_TROPO','BIO_PH','BIO_PAO2',
    'BIO_PACO2','BIO_BICARB'],
}


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2
import statsmodels.api as sm
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from scipy.stats import spearmanr
from xgboost import XGBClassifier
from econml.dml import CausalForestDML

np.random.seed(SEED)
RESULTS = []      # one row per dataset/outcome (cross-fitted)


## Loaders (self-contained, restricted to the curated DML columns)

Each dataset is loaded and cohort-filtered exactly as its `*Util`/`*AnalysisDML` does, canonical
columns renamed to `TTM` / `mortality` / `neuro_favorable`, then **X is restricted to that
dataset's curated feature list** (`CURATED`). We do NOT import the `*Util.py` modules — they are
stale (wrong CSV names, a missing `top_correlations.csv`, and HYPERION uses `groupe` not a
`Hypothermia` column), which is what made the previous run fail to load PMAP/MIMIC/HYPERION.


In [ ]:
UNSCORABLE = 'Unable to score due to medication'

def _mgcs_gap_ok(dt_first, dt_last, name):
    gap = (dt_last - dt_first).abs()
    keep = gap >= MIN_MGCS_GAP.get(name, 0)
    if MIN_MGCS_GAP.get(name, 0) > 0:
        print(f"  [{name}] mGCS time-gap filter (>= {MIN_MGCS_GAP[name]}): "
              f"excluding {(~keep).sum()} of {len(keep)} (median gap {gap.median():.0f})")
    return keep

def load_eicu():
    df = pd.read_csv(CSV['eICU'])
    f = (df['LastMGCS'] != UNSCORABLE) & (~df['LastMGCS'].isna())
    f = f & (df['FirstMGCSTime'] != df['LastMGCSTime'])
    f = f & _mgcs_gap_ok(df['FirstMGCSTime'], df['LastMGCSTime'], 'eICU')   # leakage guard
    for c in ['FirstGCS', 'FirstMGCS', 'LastMGCS', 'LastGCS']:
        df.loc[df[c] == UNSCORABLE, c] = np.nan
    df.loc[df['DeathAtDischarge'] == 1, 'LastMGCS'] = 1
    df['gender'] = (df['gender'] == 'Male').astype(int)
    df.loc[f, 'LastMGCSPositive'] = (df.loc[f, 'LastMGCS'].astype(float) == 6).astype(int)
    df = df[f & (df['nurse_first_Motor'] != 6) & ~df['Hypothermia'].isna()].copy()
    return df.rename(columns={'Hypothermia': 'TTM', 'DeathAtDischarge': 'mortality',
                              'LastMGCSPositive': 'neuro_favorable'})

def _load_epic(path, name):
    df = pd.read_csv(path)
    f = (df['first_mGCS_time'] != df['last_mGCS_time'])
    f = f & _mgcs_gap_ok(df['first_mGCS_time'], df['last_mGCS_time'], name)   # leakage guard
    df.loc[df['death_at_disch'] == 1, 'last_mGCS'] = 1
    df.loc[f, 'LastMGCSPositive'] = (df.loc[f, 'last_mGCS'].astype(float) == 6).astype(int)
    df = df[f & (df['first_mGCS'] != 6) & ~df['hypothermia'].isna()].copy()
    return df.rename(columns={'hypothermia': 'TTM', 'death_at_disch': 'mortality',
                              'LastMGCSPositive': 'neuro_favorable'})

def load_pmap():  return _load_epic(CSV['PMAP'], 'PMAP')
def load_mimic(): return _load_epic(CSV['MIMIC-IV'], 'MIMIC-IV')

def load_hyperion():
    df = pd.read_csv(CSV['HYPERION'])
    df = df[df['groupe'] != 2].copy()                 # drop non-randomized/excluded arm
    df['TTM'] = (df['groupe'] == 1).astype(int)        # 1 = hypothermia, 0 = normothermia
    return df.rename(columns={'hospital_mortality': 'mortality', 'CPC12': 'neuro_favorable'})

LOADERS = {'eICU': load_eicu, 'PMAP': load_pmap, 'MIMIC-IV': load_mimic, 'HYPERION': load_hyperion}

def load_full(name, outcome_col):
    df = LOADERS[name]()
    cand = [c for c in CURATED[name] if c in df.columns]
    X = df[cand].apply(pd.to_numeric, errors='coerce')
    T = pd.to_numeric(df['TTM'], errors='coerce')
    y = pd.to_numeric(df[outcome_col], errors='coerce')
    m = (T.notna() & y.notna()).values
    return (X[m].reset_index(drop=True), T[m].astype(int).reset_index(drop=True),
            y[m].astype(int).reset_index(drop=True))


def preprocess(X_tr, X_te):
    """Scale numeric (>2 levels) on train (StandardScaler is NaN-safe), KNN-impute (k=10) on
    train; apply to test. Column set is fixed (keep_empty_features) so out-of-fold rows align."""
    cols = list(X_tr.columns)
    num = [c for c in cols if X_tr[c].dropna().nunique() > 2]
    scaler = StandardScaler().fit(X_tr[num])
    imputer = KNNImputer(n_neighbors=10, keep_empty_features=True)

    def tf(X, fit=False):
        X = X.copy()
        if num:
            X[num] = scaler.transform(X[num])
        arr = imputer.fit_transform(X) if fit else imputer.transform(X)
        return pd.DataFrame(arr, columns=cols, index=X.index)

    return tf(X_tr, fit=True), tf(X_te)


def _fit_eval(Xtr, Xte, Ttr, ytr, observational):
    cf = fit_cf(Xtr, Ttr, ytr)
    cate = np.ravel(cf.effect(Xte))
    lo, hi = cf.effect_interval(Xte, alpha=0.05)
    ps = cf_propensity(cf, Xte) if observational else np.full(len(Xte), np.nan)
    risk = LogisticRegression(max_iter=5000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    return cate, np.ravel(lo), np.ravel(hi), ps, risk

def crossfit(name, outcome_col, observational, cols=None):
    """HTE-estimation set. EVAL_METHOD='crossfit' predicts every patient's CATE out-of-fold
    (full n, not seed-dependent); EVAL_METHOD='split' uses a single stratified 70/30 train/test
    (matches the main paper) and returns the held-out test rows. `cols` restricts the features."""
    X, T, y = load_full(name, outcome_col)
    if cols is not None:
        X = X[[c for c in cols if c in X.columns]]
    strat = (y.astype(str) + '_' + T.astype(str)).values

    if EVAL_METHOD == 'split':
        tr, te = train_test_split(np.arange(len(y)), test_size=TEST_SIZE,
                                  random_state=SEED, stratify=strat)
        Xtr, Xte = preprocess(X.iloc[tr], X.iloc[te])
        cate, lo, hi, ps, risk = _fit_eval(Xtr, Xte, T.iloc[tr], y.iloc[tr], observational)
        return dict(X=Xte.reset_index(drop=True), T=T.iloc[te].values, y=y.iloc[te].values,
                    n_features=Xte.shape[1], cate=cate, lo=lo, hi=hi, ps=ps, risk=risk)

    n = len(y)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof = {k: np.full(n, np.nan) for k in ['cate', 'ps', 'risk', 'lo', 'hi']}
    oof_X = pd.DataFrame(np.nan, index=range(n), columns=X.columns, dtype=float)
    for tr, te in skf.split(X, strat):
        Xtr, Xte = preprocess(X.iloc[tr], X.iloc[te])
        oof_X.iloc[te] = Xte.values
        cate, lo, hi, ps, risk = _fit_eval(Xtr, Xte, T.iloc[tr], y.iloc[tr], observational)
        oof['cate'][te] = cate; oof['lo'][te] = lo; oof['hi'][te] = hi
        oof['ps'][te] = ps; oof['risk'][te] = risk
    return dict(X=oof_X, T=T.values, y=y.values, n_features=oof_X.shape[1], **oof)


## Shared machinery (identical to the pooled notebook)

`ipw_interaction_test` and `run_gates` are the copy-paste-ready functions referenced in the
response letter; they match the paper's GATES/propensity handling.


In [ ]:
def fit_cf(X_tr, T_tr, y_tr, seed=SEED):
    cf = CausalForestDML(
        model_y=XGBClassifier(max_depth=3, n_estimators=50, random_state=seed),
        model_t=XGBClassifier(max_depth=2, n_estimators=20, random_state=seed),
        discrete_treatment=True, discrete_outcome=True,
        random_state=seed, n_jobs=-1)
    cf.fit(y_tr, T_tr, X=X_tr, cache_values=True)
    return cf


def cf_propensity(cf, X):
    preds = []
    for mc in cf.models_t:
        for mdl in mc:
            p = mdl.predict_proba(np.asarray(X))
            preds.append(p[:, 1] if p.ndim == 2 else np.ravel(p))
    return np.clip(np.mean(np.vstack(preds), axis=0), 1e-6, 1 - 1e-6)


def lrt_cate_interaction(y, T, cate):
    df = pd.DataFrame({'const': 1.0, 'T': np.asarray(T, float), 'cate': np.asarray(cate, float)})
    df['tx'] = df['T'] * df['cate']
    y = np.asarray(y, float)
    if np.ptp(df['cate']) < 1e-12:      # CATE constant -> no interaction estimable
        return {'lr_stat': np.nan, 'p': np.nan, 'note': 'CATE constant (TTM unused)'}
    m0 = sm.Logit(y, df[['const', 'T', 'cate']]).fit(disp=False)
    m1 = sm.Logit(y, df[['const', 'T', 'cate', 'tx']]).fit(disp=False)
    lr = 2 * (m1.llf - m0.llf)
    return {'lr_stat': lr, 'p': chi2.sf(lr, 1), 'note': ''}


def ipw_interaction_test(y, T, cate, ps, clip=PS_CLIP):
    # IPW weights with a ROBUST sandwich SE (cov_type='HC1') for the Wald test. var_weights
    # (the old approach) treats IPW weights as known precisions and understates the SE ->
    # anti-conservative; robust SE is the correct inference for weighted estimation.
    T = np.asarray(T, float); y = np.asarray(y, float)
    ps = np.clip(np.asarray(ps, float), *clip)
    w = np.where(T == 1, 1.0 / ps, 1.0 / (1.0 - ps))
    df = pd.DataFrame({'const': 1.0, 'T': T, 'cate': np.asarray(cate, float)})
    df['tx'] = df['T'] * df['cate']
    if np.ptp(df['cate']) < 1e-12:
        return {'wald_p': np.nan, 'wald_p_naive': np.nan}
    fam = sm.families.Binomial()
    m1 = sm.GLM(y, df[['const', 'T', 'cate', 'tx']], family=fam,
                freq_weights=w).fit(cov_type='HC1')
    m1_naive = sm.GLM(y, df[['const', 'T', 'cate', 'tx']], family=fam, var_weights=w).fit()
    return {'wald_p': m1.pvalues['tx'], 'wald_p_naive': m1_naive.pvalues['tx']}


def run_gates(cate, y, T, ps, observational, title='', plot=True):
    d = pd.DataFrame({'y': np.asarray(y, float), 'T': np.asarray(T, float),
                      'cate': np.asarray(cate, float)})
    if observational:
        ps = np.clip(np.asarray(ps, float), *PS_CLIP)
        d['w'] = np.where(d['T'] == 1, 1 / ps, 1 / (1 - ps))
    else:
        d['w'] = 1.0                      # HYPERION: unweighted OLS, as in Fig 3
    d['g'] = pd.qcut(d['cate'], q=N_GATES_GROUPS, labels=False, duplicates='drop') + 1
    rows = []
    for g, sub in d.groupby('g'):
        wls = sm.WLS(sub['y'], sm.add_constant(sub['T']), weights=sub['w']).fit()
        ci = np.asarray(wls.conf_int())
        rows.append({'group': int(g), 'n': len(sub), 'gate': wls.params.iloc[1],
                     'ci_low': ci[1, 0], 'ci_high': ci[1, 1]})
    gdf = pd.DataFrame(rows)
    gd = pd.get_dummies(d['g'].astype(int), prefix='g', drop_first=True).astype(float)
    Xr = sm.add_constant(pd.concat([d[['T']], gd], axis=1))
    Xf = Xr.copy()
    for c in gd.columns:
        Xf[f'T_x_{c}'] = d['T'].values * gd[c].values
    mf = sm.WLS(d['y'], Xf, weights=d['w']).fit()
    mr = sm.WLS(d['y'], Xr, weights=d['w']).fit()
    f_stat, f_p, _ = mf.compare_f_test(mr)
    rho, rho_p = stats.spearmanr(gdf['group'], gdf['gate'])
    if plot:
        fig, ax = plt.subplots(figsize=(5.5, 4))
        ax.errorbar(gdf['group'], gdf['gate'],
                    yerr=[gdf['gate'] - gdf['ci_low'], gdf['ci_high'] - gdf['gate']],
                    fmt='o-', capsize=5)
        ax.axhline(0, ls='--', color='gray')
        ax.set_xlabel('CATE quintile'); ax.set_ylabel('GATE')
        ax.set_title(f'GATES (neuro) — {title}\nF p={f_p:.3f}, rho={rho:.2f} (p={rho_p:.3f})')
        plt.tight_layout()
        plt.savefig(f"gates_neuro_{title.lower().replace(' ', '_').replace('-', '')}.png", dpi=150)
        plt.show()
    return gdf, {'f_p': f_p, 'spearman_rho': rho, 'spearman_p': rho_p}


def lasso_slearner(X_tr, T_tr, y_tr, seed=SEED):
    """L1 S-learner on curated features. Cs is an explicit *penalizing* grid (small C = strong
    penalty). On the curated ~30 features this converges and the surviving-interaction count is
    meaningful (0 => no heterogeneity survives regularization)."""
    Xd = X_tr.copy()
    Xd['TTM'] = np.asarray(T_tr, float)
    inter = []
    for c in X_tr.columns:
        ic = f'{c}_x_TTM'
        Xd[ic] = Xd[c] * Xd['TTM']
        inter.append(ic)
    clf = LogisticRegressionCV(penalty='l1', solver='saga', Cs=[1e-3, 1e-2, 1e-1, 1.0],
                               cv=5, scoring='neg_log_loss', max_iter=20000,
                               n_jobs=-1, random_state=seed)
    clf.fit(Xd, np.asarray(y_tr))
    coefs = pd.Series(clf.coef_[0], index=Xd.columns)
    nz = coefs[inter][coefs[inter] != 0]
    return len(nz), len(inter), float(clf.C_[0]), nz


## Confounding-by-indication diagnostic (mortality)

Tests whether the observational mortality CATE signal reflects residual confounding rather
than a true effect. If the estimated per-patient TTM effect on mortality **tracks baseline
mortality risk** (i.e. TTM 'looks more harmful' precisely in the sickest), and if within the
top-CATE quintile the **treated patients were sicker at baseline** than the untreated, that is
the signature of confounding by indication (sicker patients preferentially cooled). Baseline
risk is a logistic P(death | covariates) fit on the training set (treatment-blind).


In [ ]:
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression

def confounding_diagnostic(Xtr, y_tr, Xte, T_te, cate, ps, name, verbose=True):
    T_te = np.asarray(T_te)
    base = LogisticRegression(max_iter=5000).fit(Xtr, np.asarray(y_tr))
    risk = base.predict_proba(Xte)[:, 1]                    # baseline mortality risk (test)

    rho, prho = spearmanr(cate, risk)                        # CATE vs baseline risk
    q = pd.qcut(pd.Series(cate).rank(method='first'), 5, labels=False)
    top = (q == 4).values
    r_ttm  = risk[top & (T_te == 1)].mean() if (top & (T_te == 1)).any() else np.nan
    r_noap = risk[top & (T_te == 0)].mean() if (top & (T_te == 0)).any() else np.nan
    ps_top = np.asarray(ps)[top]
    frac_extreme = float(np.mean((ps_top < 0.1) | (ps_top > 0.9)))

    if verbose:
        print(f"  [confounding] CATE~baseline-risk Spearman rho={rho:.2f}; "
              f"top-quintile baseline risk TTM={r_ttm:.2f} vs no-TTM={r_noap:.2f} "
              f"(diff={r_ttm - r_noap:+.2f}); extreme-propensity in stratum={frac_extreme:.0%}")
    return {'conf_cate_risk_rho': rho, 'conf_top_risk_ttm': r_ttm, 'conf_top_risk_nottm': r_noap,
            'conf_top_risk_diff': r_ttm - r_noap, 'conf_top_frac_extreme_ps': frac_extreme}


## Run — datasets × outcomes

For each dataset and outcome: fit the dataset's own CausalForest, then LRT, IPW test
(observational only), GATES (neuro only, since Fig 3 already covers mortality GATES), and
the lasso S-learner. Observational mortality analyses also run the confounding diagnostic.
eICU additionally gets a uniform |r|>0.7 filter variant.


In [ ]:
def run_one(cfg, outcome_key):
    outcome_col = 'mortality' if outcome_key == 'mortality' else 'neuro_favorable'
    label = f"{cfg['name']} / {outcome_key}"
    try:
        cv = crossfit(cfg['name'], outcome_col, cfg['observational'])
    except Exception as e:
        print(f"[SKIP] {label}: {type(e).__name__}: {e}")
        RESULTS.append({'dataset': cfg['name'], 'outcome': outcome_key, 'status': f'fail: {e}'})
        return
    cate, T, y = cv['cate'], cv['T'], cv['y']
    pct_ci = float(np.mean((cv['lo'] < 0) & (cv['hi'] > 0)))
    row = {'dataset': cfg['name'], 'outcome': outcome_key, 'status': 'ok',
           'n': len(y), 'n_features': cv['n_features'],
           'lrt_p': lrt_cate_interaction(y, T, cate)['p'], 'pct_cate_ci_cross_zero': pct_ci}

    if cfg['observational']:
        row['ipw_wald_p'] = ipw_interaction_test(y, T, cate, cv['ps'])['wald_p']
        if outcome_key == 'mortality':                      # confounding read on out-of-fold risk
            risk = cv['risk']
            row['conf_cate_risk_rho'] = spearmanr(cate, risk)[0]
            top = (pd.qcut(pd.Series(cate).rank(method='first'), 5, labels=False) == 4).values
            row['conf_top_risk_diff'] = (risk[top & (T == 1)].mean() - risk[top & (T == 0)].mean())
    if outcome_key == 'neuro':
        _, g = run_gates(cate, y, T, cv['ps'], cfg['observational'], title=cfg['name'])
        row['gates_neuro_f_p'] = g['f_p']; row['gates_neuro_spearman_rho'] = g['spearman_rho']

    # lasso S-learner on the (out-of-fold) full design, internal CV for the penalty
    n_nz, n_int, C, _ = lasso_slearner(cv['X'], T, y)
    row['lasso_nonzero_interactions'] = n_nz; row['lasso_n_interactions'] = n_int
    RESULTS.append(row)

    print(f"=== {label}: n={len(y)}, {cv['n_features']} features, {EVAL_METHOD} ===")
    msg = f"  LRT p={row['lrt_p']:.4f}"
    if 'ipw_wald_p' in row: msg += f"; IPW p={row['ipw_wald_p']:.4f}"
    if 'gates_neuro_f_p' in row: msg += f"; GATES-neuro F p={row['gates_neuro_f_p']:.4f}"
    msg += f"; lasso {n_nz}/{n_int}"
    print(msg)
    if 'conf_top_risk_diff' in row:
        print(f"  [confounding] CATE~baseline-risk rho={row['conf_cate_risk_rho']:+.2f}; "
              f"top-quintile treated-minus-untreated baseline risk={row['conf_top_risk_diff']:+.2f}")


for cfg in DATASETS:
    for outcome_key in ['mortality', 'neuro']:
        run_one(cfg, outcome_key)


## eICU uniform collinearity filter (R1 minor d)

Reviewer 1 asked why balance is tighter in PMAP/MIMIC (which drop covariates with
|r|>0.7 vs TTM) than in eICU (which does not). Here we apply the same filter to eICU and
confirm the HTE conclusion is unchanged.


In [ ]:
def run_eicu_uniform_filter(outcome_key='mortality'):
    outcome_col = 'mortality' if outcome_key == 'mortality' else 'neuro_favorable'
    X, T, y = load_full('eICU', outcome_col)
    corr = X.apply(lambda c: np.corrcoef(c.fillna(c.median()), T)[0, 1]).abs()
    drop = corr[corr > CORR_THRESHOLD].index.tolist()
    keep = [c for c in X.columns if c not in drop]
    print(f"eICU |r|>{CORR_THRESHOLD} with TTM -> dropping {len(drop)}: {drop}")
    cv = crossfit('eICU', outcome_col, True, cols=keep)
    lrt = lrt_cate_interaction(cv['y'], cv['T'], cv['cate'])
    ipw = ipw_interaction_test(cv['y'], cv['T'], cv['cate'], cv['ps'])
    print(f"  after filter: LRT p={lrt['p']:.4f}, IPW p={ipw['wald_p']:.4f}")
    RESULTS.append({'dataset': 'eICU (|r|>0.7 filter)', 'outcome': outcome_key, 'status': 'ok',
                    'n_features': len(keep), 'lrt_p': lrt['p'], 'ipw_wald_p': ipw['wald_p'],
                    'lrt_note': f'dropped {len(drop)} collinear'})

run_eicu_uniform_filter('mortality')


## Summary for the response letter

In [ ]:
summary = pd.DataFrame(RESULTS)
summary.to_csv('per_dataset_results.csv', index=False)
display(summary.round(4))

print('\n--- Paste-ready ---')
ok = summary[summary['status'] == 'ok']
for _, r in ok.iterrows():
    bits = [f"LRT p={r.get('lrt_p')}"]
    if pd.notna(r.get('ipw_wald_p')):        bits.append(f"IPW p={r.get('ipw_wald_p'):.3f}")
    if pd.notna(r.get('gates_neuro_f_p')):   bits.append(f"GATES-neuro F p={r.get('gates_neuro_f_p'):.3f}")
    if pd.notna(r.get('lasso_nonzero_interactions')):
        bits.append(f"lasso {int(r['lasso_nonzero_interactions'])}/{int(r['lasso_n_interactions'])} interactions")
    if pd.notna(r.get('conf_cate_risk_rho')):
        bits.append(f"CATE~risk rho={r['conf_cate_risk_rho']:.2f}; top-quintile risk TTM-noTTM="
                    f"{r['conf_top_risk_diff']:+.2f}")
    print(f"{r['dataset']:24s} {r['outcome']:9s}: " + '; '.join(str(b) for b in bits))

print('\n--- Confounding-by-indication read (mortality, observational) ---')
print('A positive CATE~risk rho (harm predicted in sicker patients) AND a positive')
print('top-quintile TTM-minus-noTTM baseline-risk difference (treated were sicker) together')
print('indicate the mortality interaction reflects confounding by indication, not true HTE.')
